# LIL vs triplet sparse assembly for ED

This notebook benchmarks two Hamiltonian builders with the same operator logic:

- `hamiltonian_lil.py`: incremental insertion into `scipy.sparse.lil_matrix`
- `hamiltonian_triplet.py`: append `(row, col, value)` triplets, then build COO and convert to CSR

It also checks that the resulting matrices are identical.

In [ ]:
import time
from statistics import mean, stdev

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse.linalg import eigsh

from basis import FullBasis, SzBasis
from hamiltonian_lil import SpinChainHamiltonian as LILHamiltonian
from hamiltonian_triplet import SpinChainHamiltonian as TripletHamiltonian


In [ ]:
def build_model(builder_cls, basis, model='heisenberg', **kwargs):
    Hbuilder = builder_cls(basis)
    if model == 'heisenberg':
        Hbuilder.add_heisenberg(kwargs.get('Jxy', 1.0), kwargs.get('Jz', 1.0), periodic=kwargs.get('periodic', True))
    elif model == 'xxz_zfield':
        Hbuilder.add_heisenberg(kwargs.get('Jxy', 1.0), kwargs.get('Jz', 1.0), periodic=kwargs.get('periodic', True))
        Hbuilder.add_z_field(kwargs.get('h', 0.3))
    elif model == 'tfim_like':
        Hbuilder.add_zz(kwargs.get('Jz', 1.0), periodic=kwargs.get('periodic', True))
        Hbuilder.add_x_field(kwargs.get('g', 0.7))
    else:
        raise ValueError(f'Unknown model: {model}')
    return Hbuilder.build()

def benchmark(builder_cls, basis_factory, sizes, repeats=3, model='heisenberg', **kwargs):
    records = []
    for L in sizes:
        basis = basis_factory(L)
        times = []
        nnz = None
        for _ in range(repeats):
            t0 = time.perf_counter()
            H = build_model(builder_cls, basis, model=model, **kwargs)
            dt = time.perf_counter() - t0
            times.append(dt)
            nnz = H.nnz
        records.append({
            'L': L,
            'dim': basis.dim,
            'nnz': nnz,
            'time_mean_s': mean(times),
            'time_std_s': stdev(times) if len(times) > 1 else 0.0,
            'builder': builder_cls.__name__,
            'basis_name': basis.__class__.__name__,
        })
    return pd.DataFrame(records)


In [ ]:
# Quick correctness check on one FullBasis case and one SzBasis case
for basis in [FullBasis(8), SzBasis(10, 5)]:
    H_lil = build_model(LILHamiltonian, basis, model='heisenberg', Jxy=1.0, Jz=1.0)
    H_trip = build_model(TripletHamiltonian, basis, model='heisenberg', Jxy=1.0, Jz=1.0)
    diff = (H_lil - H_trip).tocoo()
    print(type(basis).__name__, basis.dim, diff.nnz, np.max(np.abs(diff.data)) if diff.nnz else 0.0)


## Benchmarks

I benchmark two basis choices:

- `FullBasis(L)` for small sizes
- `SzBasis(L, L//2)` near half filling

The half-filling sector is usually the more relevant sparse ED case.

In [ ]:
full_sizes = [8, 9, 10, 11, 12]
sz_sizes = [10, 12, 14, 16]
repeats = 3

df_full_lil = benchmark(LILHamiltonian, lambda L: FullBasis(L), full_sizes, repeats=repeats, model='heisenberg')
df_full_trip = benchmark(TripletHamiltonian, lambda L: FullBasis(L), full_sizes, repeats=repeats, model='heisenberg')

df_sz_lil = benchmark(LILHamiltonian, lambda L: SzBasis(L, L//2), sz_sizes, repeats=repeats, model='heisenberg')
df_sz_trip = benchmark(TripletHamiltonian, lambda L: SzBasis(L, L//2), sz_sizes, repeats=repeats, model='heisenberg')

df_full = pd.concat([df_full_lil.assign(method='LIL'), df_full_trip.assign(method='Triplet')], ignore_index=True)
df_sz = pd.concat([df_sz_lil.assign(method='LIL'), df_sz_trip.assign(method='Triplet')], ignore_index=True)

display(df_full[['L', 'dim', 'nnz', 'method', 'time_mean_s', 'time_std_s']])
display(df_sz[['L', 'dim', 'nnz', 'method', 'time_mean_s', 'time_std_s']])


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
for method, sub in df_full.groupby('method'):
    ax.errorbar(sub['dim'], sub['time_mean_s'], yerr=sub['time_std_s'], marker='o', label=method)
ax.set_xlabel('Hilbert-space dimension')
ax.set_ylabel('Build time [s]')
ax.set_title('Full basis: Heisenberg assembly time')
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
for method, sub in df_sz.groupby('method'):
    ax.errorbar(sub['dim'], sub['time_mean_s'], yerr=sub['time_std_s'], marker='o', label=method)
ax.set_xlabel('Sector dimension')
ax.set_ylabel('Build time [s]')
ax.set_title('Half-filling Sz basis: Heisenberg assembly time')
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()


In [ ]:
# Relative speedup > 1 means triplet is faster
df_compare = df_sz.pivot(index='L', columns='method', values='time_mean_s').reset_index()
df_compare['LIL_over_Triplet'] = df_compare['LIL'] / df_compare['Triplet']
display(df_compare)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(df_compare['L'], df_compare['LIL_over_Triplet'], marker='o')
ax.axhline(1.0, linestyle='--')
ax.set_xlabel('L')
ax.set_ylabel('time(LIL) / time(Triplet)')
ax.set_title('Relative speed in the half-filling Sz sector')
ax.grid(True, alpha=0.3)
plt.show()


## Optional: tiny diagonalization sanity check

This is not part of the assembly benchmark. It just confirms both builders produce the same low-energy spectrum.

In [ ]:
basis = SzBasis(12, 6)
H_lil = build_model(LILHamiltonian, basis, model='heisenberg', Jxy=1.0, Jz=1.0)
H_trip = build_model(TripletHamiltonian, basis, model='heisenberg', Jxy=1.0, Jz=1.0)

evals_lil = eigsh(H_lil, k=4, which='SA', return_eigenvectors=False)
evals_trip = eigsh(H_trip, k=4, which='SA', return_eigenvectors=False)

print('LIL   :', np.sort(evals_lil))
print('Triplet:', np.sort(evals_trip))
print('max |ΔE| =', np.max(np.abs(np.sort(evals_lil) - np.sort(evals_trip))))
